In [1]:
import warnings

import joblib
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv("house_prices.csv")

In [3]:
df.columns

Index(['Index', 'Title', 'Description', 'Amount(in rupees)',
       'Price (in rupees)', 'location', 'Carpet Area', 'Status', 'Floor',
       'Transaction', 'Furnishing', 'facing', 'overlooking', 'Society',
       'Bathroom', 'Balcony', 'Car Parking', 'Ownership', 'Super Area',
       'Dimensions', 'Plot Area'],
      dtype='object')

In [4]:
df.head()

,Index,Title,Description,Amount(in rupees),Price (in rupees),location,Carpet Area,Status,Floor,Transaction,...,facing,overlooking,Society,Bathroom,Balcony,Car Parking,Ownership,Super Area,Dimensions,Plot Area
0,0,1 BHK Ready to Occupy Flat for sale in Srushti...,"Bhiwandi, Thane has an attractive 1 BHK Flat f...",42 Lac,6000.0,thane,500 sqft,Ready to Move,10 out of 11,Resale,...,NaN,NaN,Srushti Siddhi Mangal Murti Complex,1,2,NaN,NaN,NaN,NaN,NaN
1,1,2 BHK Ready to Occupy Flat for sale in Dosti V...,One can find this stunning 2 BHK flat for sale...,98 Lac,13799.0,thane,473 sqft,Ready to Move,3 out of 22,Resale,...,East,Garden/Park,Dosti Vihar,2,NaN,1 Open,Freehold,NaN,NaN,NaN
2,2,2 BHK Ready to Occupy Flat for sale in Sunrise...,Up for immediate sale is a 2 BHK apartment in ...,1.40 Cr,17500.0,thane,779 sqft,Ready to Move,10 out of 29,Resale,...,East,Garden/Park,Sunrise by Kalpataru,2,NaN,1 Covered,Freehold,NaN,NaN,NaN
3,3,1 BHK Ready to Occupy Flat for sale Kasheli,This beautiful 1 BHK Flat is available for sal...,25 Lac,NaN,thane,530 sqft,Ready to Move,1 out of 3,Resale,...,NaN,NaN,NaN,1,1,NaN,NaN,NaN,NaN,NaN
4,4,2 BHK Ready to Occupy Flat for sale in TenX Ha...,"This lovely 2 BHK Flat in Pokhran Road, Thane ...",1.60 Cr,18824.0,thane,635 sqft,Ready to Move,20 out of 42,Resale,...,West,"Garden/Park, Main Road",TenX Habitat Raymond Realty,2,NaN,1 Covered,Co-operative Society,NaN,NaN,NaN


In [5]:
def convert_amount(x):
    if pd.isna(x):
        return None

    x = str(x).lower().replace(",", "").strip()

    if "lac" in x:
        inr = float(x.replace("lac", "").strip()) * 100000
        return float(f"{inr / 93:.4f}")
        
    elif "cr" in x:
        inr = float(x.replace("cr", "").strip()) * 10000000
        return float(f"{inr / 93:.4f}")

    elif "call" in x:
        return None
        
    else:
        try:
            return float(x)
        except:
            return None

In [6]:
def convert_carpet_area(x):
    if pd.isna(x):
        return None
    x = str(x).lower()
    
    if "sqft" in x:
        m2 = float(x.replace("sqft", "").strip()) *  0.092903
        return float(f"{m2:.4f}")
        
    elif "sqyrd" in x:
        m2 = float(x.replace("sqyrd", "").strip()) * 9 *  0.092903
        return float(f"{m2:.4f}")

    else:
        try:
            return float(x)
        except:
            return None

In [7]:
def remove_outlier_from_column(df, target_cols, threshold=1.5):
    existing_cols = [col for col in target_cols if col in df.columns]

    Q1 = df[existing_cols].quantile(0.25)
    Q3 = df[existing_cols].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - threshold * IQR
    upper_bound = Q3 + threshold * IQR

    mask = ~((df[existing_cols] < lower_bound) | (df[existing_cols] > upper_bound)).any(axis=1)

    return df[mask].copy()

In [8]:
def convert_floor(x):
    if pd.isna(x):
        return None

    x = str(x).lower().strip()

    if "ground" in x:
        return 0
    elif "lower basement"  in x:
        return -1
    elif "upper basement" in x:
        return -0.5

    if "out of" in x:
        parts = x.split(" out of ")
        current_floor = int(parts[0].strip())
        total_floor = int (parts[1].strip())

        floor_ratio = current_floor / total_floor

        if floor_ratio == 0:
            return 0
        elif floor_ratio == 1:
            return 2
        elif floor_ratio > 0 and floor_ratio < 1:
            return 1
        else:
            try:
                return int(x)
            except:
                return None

In [9]:
def eda_preprocess(df) -> pd.DataFrame:

    df = df.copy()

    df.columns = (df.columns
                  .str.strip()
                  .str.lower()
                  .str.replace(" ", "_")
                 )
    df = df.rename(columns={
        "amount(in_rupees)":"amount", "price_(in_rupees)":"price"
    })
    
    df.drop([
        "index", "description", "society", "title", "dimensions", "super_area", "plot_area"
    ], axis=1, inplace=True)

    #====Feature engineering===
    df["has_parking"] = df["car_parking"].notna().astype(int)
    df.drop(["car_parking"], axis=1, inplace=True)

    for i in ["bathroom", "balcony"]:
        if i in df.columns:
            df[i] = df[i].astype(str)
            df[i] = df[i].replace("> 10", "11")
            df[i] = df[i].replace("nan", np.nan)
            df[i] = df[i].astype(float)
        
    df["amount"] = df["amount"].apply(convert_amount)
    df["carpet_area"] = df["carpet_area"].apply(convert_carpet_area)
    df["floor"] = df["floor"].apply(convert_floor)

    df = remove_outlier_from_column(df, ["amount", "carpet_area"], threshold=1)
    df = remove_outlier_from_column(df, ["bathroom", "balcony"], threshold=2)
    df = remove_outlier_from_column(df, ["price"], threshold=2)

    df = df[(df["price"].notna()) & (df["price"] > 0)]

    return df

In [10]:
df = eda_preprocess(df)

In [11]:
X = df.drop("price", axis=1)
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=42)

In [12]:
class HousePreprocessor(BaseEstimator, TransformerMixin):

    def __init__(self):
        pass
    
    def fit(self, X, y=None):
        
        num_cols = X.select_dtypes(["int64", "float64"]).columns
        cat_cols = X.select_dtypes(["object"]).columns

        self.num_medians_ = { # sona '_' koymak -> fit sirasinda ogrenildi anlami tasir. 
            col : X[col].median() for col in num_cols
        }     

        self.cat_mode_ = {
            col : X[col].mode()[0] for col in cat_cols
        }
        
        self.top_locations_ = X["location"].value_counts().nlargest(20).index
        
        return self
    
    def transform(self, X):
        X = X.copy()
        
        for col, val in self.num_medians_.items():
            if col in X.columns:
                X[col] = X[col].fillna(val)
            
        for col, val in self.cat_mode_.items():
            if col in X.columns:
                X[col] = X[col].fillna(val)
        
        if "furnishing" in X.columns:
            X["furnishing"] = X["furnishing"].map({"Unfurnished": 0, "Semi-Furnished": 1, "Furnished": 2})

        if "location" in X.columns:
            X["location"] = X["location"].apply(lambda x: x if x in self.top_locations_ else "other")
    
        return X

In [13]:
cat_cols = df.select_dtypes(["object"]).columns

pipeline = Pipeline([
    ("preprocessor", HousePreprocessor()),
    
    ("encoder", ColumnTransformer(
    [   
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
    ], remainder="passthrough")
    ),
    
    ("model", XGBRegressor())
])

In [14]:
pipeline.fit(X_train, y_train)
y_test_pred = pipeline.predict(X_test)
y_train_pred = pipeline.predict(X_train)

In [15]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2 = r2_score(true, predicted)

    return mae, rmse, r2

In [16]:
def model_score_table(y_test_pred, y_train_pred, name):
    table = []
    
    y_test_mae, y_test_rmse, y_test_r2 = evaluate_model(y_test, y_test_pred)
    y_train_mae, y_train_rmse, y_train_r2 = evaluate_model(y_train, y_train_pred)
    
    table.append({
        "Model"                  : name,
        "Mean Absolute Error"    : y_test_mae,
        "Root Mean Square Error" : y_test_rmse,
        "R2 Score"               : y_test_r2,
        "Overfitting"            : y_train_r2 - y_test_r2,
        "Ratio"                  : y_test_rmse / y_train_rmse 
    })
    
    return pd.DataFrame(table).set_index("Model")  

In [17]:
model_score_table(y_test_pred, y_train_pred, name="XGB") 

,Mean Absolute Error,Root Mean Square Error,R2 Score,Overfitting,Ratio
Model,,,,,
XGB,266.713061,573.242425,0.955247,0.009826,1.129657


In [18]:
df.tail()

,amount,price,location,carpet_area,status,floor,transaction,furnishing,facing,overlooking,bathroom,balcony,ownership,has_parking
187526,67741.9355,3225.0,zirakpur,NaN,Ready to Move,1.0,New Property,Semi-Furnished,East,Garden/Park,3.0,3.0,Freehold,1
187527,59139.7849,3274.0,zirakpur,NaN,Ready to Move,1.0,Resale,Unfurnished,North - East,"Garden/Park, Main Road",3.0,NaN,NaN,1
187528,81720.4301,4343.0,zirakpur,116.1287,Ready to Move,1.0,Resale,Furnished,East,"Garden/Park, Main Road",3.0,2.0,Freehold,1
187529,32258.0645,4231.0,zirakpur,NaN,Ready to Move,2.0,Resale,Semi-Furnished,NaN,Main Road,2.0,NaN,NaN,0
187530,126881.7204,6162.0,zirakpur,NaN,Ready to Move,1.0,Resale,Semi-Furnished,North - East,"Garden/Park, Pool",4.0,4.0,Freehold,1


In [19]:
df["location"].unique()

array(['thane', 'navi-mumbai', 'nagpur', 'mumbai', 'ahmedabad',
       'bangalore', 'chennai', 'gurgaon', 'hyderabad', 'indore', 'jaipur',
       'kolkata', 'lucknow', 'new-delhi', 'noida', 'pune', 'agra',
       'ahmadnagar', 'allahabad', 'aurangabad', 'badlapur', 'belgaum',
       'bhiwadi', 'bhiwandi', 'bhopal', 'bhubaneswar', 'chandigarh',
       'coimbatore', 'dehradun', 'durgapur', 'ernakulam', 'faridabad',
       'ghaziabad', 'goa', 'greater-noida', 'guntur', 'guwahati',
       'gwalior', 'haridwar', 'jabalpur', 'jamshedpur', 'jodhpur',
       'kalyan', 'kanpur', 'kochi', 'kozhikode', 'ludhiana', 'madurai',
       'mangalore', 'mohali', 'mysore', 'nashik', 'navsari', 'nellore',
       'palakkad', 'palghar', 'panchkula', 'patna', 'pondicherry',
       'raipur', 'rajahmundry', 'ranchi', 'satara', 'shimla', 'siliguri',
       'solapur', 'sonipat', 'surat', 'thrissur', 'tirupati', 'trichy',
       'trivandrum', 'udaipur', 'udupi', 'vadodara', 'vapi', 'varanasi',
       'vijayawada',

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 148943 entries, 0 to 187530
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   amount       148943 non-null  float64
 1   price        148943 non-null  float64
 2   location     148943 non-null  object 
 3   carpet_area  77049 non-null   float64
 4   status       148371 non-null  object 
 5   floor        142618 non-null  float64
 6   transaction  148937 non-null  object 
 7   furnishing   146988 non-null  object 
 8   facing       91539 non-null   object 
 9   overlooking  83092 non-null   object 
 10  bathroom     148214 non-null  float64
 11  balcony      106144 non-null  float64
 12  ownership    94412 non-null   object 
 13  has_parking  148943 non-null  int64  
dtypes: float64(6), int64(1), object(7)
memory usage: 17.0+ MB


In [21]:
with open("house_price_model.pkl", "wb") as f:
    joblib.dump(pipeline, f)

In [22]:
pd.DataFrame(X_test).to_csv("house_price_test_data.csv", index=False)